In [1]:
!pip install -q scanpy leidenalg igraph harmony-pytorch "pandas>=2.3" scikit-learn
# restart the runtime after this cell, then run from cell 2

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.


In [2]:
DATA_PATH = "/content/global_z0.h5ad"  # adjustable
OUT_DIR = "/content/outputs/m1_composition"  # adjustable
SEED = 0  # adjustable
SMOKE_TEST = False  # adjustable
ROBUST_SEEDS = [0, 1, 42]  # adjustable
STATE_RES = 1.0  # adjustable
STATE_NEIGHBORS = 30
N_PCS = 50
K_NEIGHBORS = 30  # adjustable
TRANSFORM = "clr"  # adjustable
PSEUDOCOUNT = 0.01
SELF_WEIGHT = False  # adjustable
SELF_WEIGHT_VALUE = 0.5
K_RANGE = list(range(4, 11))
N_INIT = 5
N_BOOT = 5
BOOT_FRAC = 0.8
SIL_SAMPLE = 5000
MIN_NICHE_FRAC = 0.005  # adjustable
TOP_N = 10
SMOKE_N = 2000  # adjustable
BASELINE = {"n_niches": 7, "ARI": 0.248, "NMI": 0.323}
if SMOKE_TEST:
    OUT_DIR = OUT_DIR + "_smoke"

In [3]:
import os, json, time, random, shutil, sys, warnings
from importlib.metadata import version, PackageNotFoundError
import numpy as np, pandas as pd, scipy.sparse as sp
import scanpy as sc, anndata as an
import numba
sc.settings.n_jobs = numba.config.NUMBA_NUM_THREADS  # match numba thread cap
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score as ARI, normalized_mutual_info_score as NMI, silhouette_score

T0 = time.time()
os.makedirs(OUT_DIR, exist_ok=True)
CONFIG_KEYS = ["DATA_PATH", "OUT_DIR", "SEED", "SMOKE_TEST", "ROBUST_SEEDS", "STATE_RES", "STATE_NEIGHBORS", "N_PCS",
               "K_NEIGHBORS", "TRANSFORM", "PSEUDOCOUNT", "SELF_WEIGHT", "SELF_WEIGHT_VALUE", "K_RANGE", "N_INIT",
               "N_BOOT", "BOOT_FRAC", "SIL_SAMPLE", "MIN_NICHE_FRAC", "TOP_N", "SMOKE_N"]

def seed_all(s):
    random.seed(s); np.random.seed(s)
    try:
        import torch
        torch.manual_seed(s)
    except ImportError:
        pass

def jdump(obj, path):
    def conv(o):
        if isinstance(o, (np.integer,)): return int(o)
        if isinstance(o, (np.floating,)): return None if np.isnan(o) else float(o)
        if isinstance(o, np.ndarray): return o.tolist()
        return str(o)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=conv)

def load_smi(path):
    ad = sc.read_h5ad(path)
    print("raw shape", ad.shape, "| obs columns:", list(ad.obs.columns))
    ad = ad[:, ~ad.var_names.str.startswith("NegPrb")].copy()
    ad = ad[ad.obs["Region"].notna()].copy()
    ad.obs_names_make_unique()
    ad.obs["Region"] = ad.obs["Region"].astype(str)
    ad.obs["batch"] = ad.obs["batch"].astype(str)
    xy = np.asarray(ad.obsm["spatial_global"])
    ad.obs["cx_g"], ad.obs["cy_g"] = xy[:, 0], xy[:, 1]
    ad.layers["counts"] = ad.X.copy()
    sc.pp.normalize_total(ad, target_sum=1e4); sc.pp.log1p(ad)
    print("after filtering", ad.shape, "| batches:", ad.obs["batch"].nunique())
    return ad

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


In [4]:
def spatial_knn(coords, batch, k):
    coords = np.asarray(coords, float); batch = np.asarray(batch); n = len(coords)
    boxes = {b: (coords[batch == b].min(0), coords[batch == b].max(0)) for b in np.unique(batch)}
    bs = list(boxes)
    for i in range(len(bs)):
        for j in range(i + 1, len(bs)):
            lo = np.maximum(boxes[bs[i]][0], boxes[bs[j]][0]); hi = np.minimum(boxes[bs[i]][1], boxes[bs[j]][1])
            if np.all(hi > lo):
                a_ov = np.prod(hi - lo)
                a_min = min(np.prod(boxes[b][1] - boxes[b][0]) for b in (bs[i], bs[j]))
                if a_ov > 0.5 * a_min:
                    warnings.warn(f"spatial_global coordinates of batches {bs[i]} and {bs[j]} overlap")
    rows, cols = [], []
    for b in bs:
        ix = np.where(batch == b)[0]
        kk = min(k, len(ix) - 1)
        if kk < 1: continue
        nn = NearestNeighbors(n_neighbors=kk + 1).fit(coords[ix])
        idx = nn.kneighbors(coords[ix], return_distance=False)
        me = idx == np.arange(len(ix))[:, None]
        me[~me.any(1), -1] = True
        nb = idx[~me].reshape(len(ix), kk)
        rows.append(np.repeat(ix, kk)); cols.append(ix[nb.ravel()])
    rows, cols = np.concatenate(rows), np.concatenate(cols)
    return sp.csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(n, n))

def state_clusters(pcs, res, k, seed):
    a = an.AnnData(np.zeros((pcs.shape[0], 1), dtype=np.float32))
    a.obsm["X_h"] = np.asarray(pcs)
    sc.pp.neighbors(a, n_neighbors=k, use_rep="X_h", random_state=seed)
    sc.tl.leiden(a, resolution=res, random_state=seed, flavor="igraph", n_iterations=2, directed=False)
    return a.obs["leiden"].astype(str).to_numpy()

def compute_states(ad, seed):
    X = ad.X.toarray() if sp.issparse(ad.X) else np.asarray(ad.X)
    a = an.AnnData(X.astype(np.float32))
    sc.pp.scale(a, max_value=10)
    sc.tl.pca(a, n_comps=N_PCS, random_state=seed)
    pcs = a.obsm["X_pca"]
    if ad.obs["batch"].nunique() > 1:
        from harmony import harmonize
        seed_all(seed)
        pcs = np.asarray(harmonize(pcs, ad.obs[["batch"]], batch_key="batch"))
    return state_clusters(pcs, STATE_RES, STATE_NEIGHBORS, seed)

In [5]:
def top_markers(ad, groups, name, n=TOP_N):
    g = pd.Series(np.asarray(groups).astype(str), index=ad.obs_names)
    valid = [x for x, c in g.value_counts().items() if c >= 2]
    mk = an.AnnData(ad.layers["counts"].copy(), obs=pd.DataFrame({"g": pd.Categorical(g.values)}, index=ad.obs_names), var=ad.var[[]].copy())
    sc.pp.normalize_total(mk, target_sum=1e4); sc.pp.log1p(mk)
    sc.tl.rank_genes_groups(mk, "g", groups=valid, reference="rest", method="wilcoxon")
    df = sc.get.rank_genes_groups_df(mk, group=None)
    df = df.rename(columns={"group": name, "names": "gene", "scores": "score", "logfoldchanges": "log2FC", "pvals_adj": "padj"})
    df["rank"] = df.groupby(name).cumcount() + 1
    df = df[df["rank"] <= n].copy()
    df["_k"] = df[name].astype(int)
    df = df.sort_values(["_k", "rank"]).drop(columns="_k").reset_index(drop=True)
    return df[[name, "rank", "gene", "log2FC", "score", "padj"]]

def evaluate(labels, region):
    lab = pd.Series(np.asarray(labels).astype(str), name="niche"); reg = pd.Series(np.asarray(region).astype(str), name="Region")
    ct = pd.crosstab(lab, reg)
    ct = ct.loc[sorted(ct.index, key=int)]
    summ = pd.DataFrame({"n_cells": ct.sum(1), "frac_cells": ct.sum(1) / ct.values.sum(),
                         "majority_region": ct.idxmax(1), "purity": ct.max(1) / ct.sum(1)})
    return {"ARI": float(ARI(reg, lab)), "NMI": float(NMI(reg, lab)), "crosstab": ct, "summary": summ}

def plot_niche_vs_truth(ad, niche, path):
    fig, axs = plt.subplots(1, 2, figsize=(14, 6))
    for ax, vals, title in [(axs[0], ad.obs["Region"].values, "expert Region"), (axs[1], np.asarray(niche), "niches")]:
        cats = sorted(set(vals), key=lambda x: (not str(x).isdigit(), int(x) if str(x).isdigit() else str(x)))
        cm = plt.get_cmap("tab20")
        for i, c in enumerate(cats):
            m = vals == c
            ax.scatter(ad.obs["cx_g"][m], ad.obs["cy_g"][m], s=1, color=cm(i % 20), label=c, rasterized=True)
        ax.set_title(title); ax.set_aspect("equal"); ax.legend(markerscale=6, fontsize=7, loc="best")
    fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)

def save_contract(out, ad, niche, extras, extra_summary, metrics, config):
    niche = np.asarray(niche).astype(str)
    lab = pd.DataFrame({"niche": niche, "Region": ad.obs["Region"].values, "cx_g": ad.obs["cx_g"].values,
                        "cy_g": ad.obs["cy_g"].values, "batch": ad.obs["batch"].values}, index=ad.obs_names)
    for c, v in extras.items(): lab[c] = np.asarray(v)
    lab.to_csv(f"{out}/labels.csv")
    long = top_markers(ad, niche, "niche")
    long.to_csv(f"{out}/niche_markers_long.csv", index=False)
    wide = long.pivot(index="rank", columns="niche", values="gene")
    wide[sorted(wide.columns, key=int)].to_csv(f"{out}/niche_markers_wide.csv")
    ev = evaluate(niche, ad.obs["Region"].values)
    summ = ev["summary"].join(extra_summary)
    summ.to_csv(f"{out}/niche_summary.csv", index_label="niche")
    ev["crosstab"].to_csv(f"{out}/crosstab_vs_region.csv")
    plot_niche_vs_truth(ad, niche, f"{out}/niche_vs_truth.png")
    jdump(metrics, f"{out}/metrics.json")
    jdump(config, f"{out}/run_config.json")
    return ev

In [6]:
def composition(W, states):
    S = sorted(set(states), key=int); m = {s: i for i, s in enumerate(S)}
    code = np.array([m[s] for s in states])
    oh = sp.csr_matrix((np.ones(len(code)), (np.arange(len(code)), code)), shape=(len(code), len(S)))
    comp = (W @ oh).toarray()
    comp = comp / np.maximum(comp.sum(1, keepdims=True), 1e-12)
    return comp, oh.toarray(), S

def features(comp, oh):
    if TRANSFORM == "clr":
        l = np.log(comp + PSEUDOCOUNT); X = l - l.mean(1, keepdims=True)
    else:
        X = np.sqrt(comp)
    return np.hstack([X, SELF_WEIGHT_VALUE * oh]) if SELF_WEIGHT else X

def mm(v):
    v = np.asarray(v, float); r = np.nanmax(v) - np.nanmin(v)
    return np.zeros_like(v) if r == 0 else (v - np.nanmin(v)) / r

def select_k(X, seed):
    # rule: mean of min-max scaled BIC-elbow (distance below the BIC chord), silhouette and 5x80% bootstrap ARI; argmax wins
    N = len(X); rng = np.random.RandomState(seed); rows = []
    subs = [rng.choice(N, int(BOOT_FRAC * N), replace=False) for _ in range(N_BOOT)]
    for k in K_RANGE:
        g = GaussianMixture(k, covariance_type="diag", n_init=N_INIT, random_state=seed).fit(X)
        lab = g.predict(X)
        sil = silhouette_score(X, lab, sample_size=min(SIL_SAMPLE, N), random_state=seed) if len(set(lab)) > 1 else np.nan
        preds = [GaussianMixture(k, covariance_type="diag", n_init=N_INIT, random_state=seed + i + 1).fit(X[s]).predict(X) for i, s in enumerate(subs)]
        aris = [ARI(preds[i][np.intersect1d(subs[i], subs[j])], preds[j][np.intersect1d(subs[i], subs[j])])
                for i in range(N_BOOT) for j in range(i + 1, N_BOOT)]
        rows.append({"k": k, "bic": g.bic(X), "silhouette": sil, "stability": float(np.mean(aris))})
        print(rows[-1])
    df = pd.DataFrame(rows)
    b = df["bic"].values; y = (b - b.min()) / (b.max() - b.min() + 1e-12)
    x = (df["k"].values - df["k"].min()) / (df["k"].max() - df["k"].min())
    df["bic_elbow"] = np.clip((y[0] + (y[-1] - y[0]) * x) - y, 0, None)
    sil = df["silhouette"].fillna(df["silhouette"].min()).values
    df["combined"] = (mm(df["bic_elbow"]) + mm(sil) + mm(df["stability"])) / 3
    return int(df["k"].values[int(df["combined"].values.argmax())]), df

def plot_selection(df, k, path):
    fig, axs = plt.subplots(1, 3, figsize=(14, 3.8))
    for ax, c in zip(axs, ["bic", "silhouette", "stability"]):
        ax.plot(df["k"], df[c], "o-"); ax.axvline(k, color="r", ls="--"); ax.set_xlabel("k"); ax.set_title(c)
    fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)

def fit_final(X, k, seed):
    g = GaussianMixture(k, covariance_type="diag", n_init=N_INIT, random_state=seed).fit(X)
    lab = g.predict(X); post = g.predict_proba(X).max(1); N = len(X)
    sizes = np.bincount(lab, minlength=k)
    small = [c for c in range(k) if sizes[c] < MIN_NICHE_FRAC * N]; keep = [c for c in range(k) if c not in small]
    merges = {}
    for c in small:
        t = keep[int(((g.means_[keep] - g.means_[c]) ** 2).sum(1).argmin())]
        lab[lab == c] = t; merges[str(c)] = str(t)
    order = pd.Series(lab).value_counts().index
    remap = {old: str(i) for i, old in enumerate(order)}
    return np.array([remap[x] for x in lab]), post, merges

def niche_composition(comp, labels, S, top=5, min_frac=0.05):
    glob = comp.mean(0); out = []
    for n in sorted(set(labels), key=int):
        d = pd.DataFrame({"niche": n, "state": S, "frac_in_niche": comp[labels == n].mean(0), "frac_global": glob})
        d["fold_enrichment"] = d["frac_in_niche"] / d["frac_global"].clip(lower=1e-12)
        d = d[d["frac_in_niche"] >= min_frac].sort_values("fold_enrichment", ascending=False).head(top).copy()
        d["rank"] = range(1, len(d) + 1); out.append(d)
    return pd.concat(out, ignore_index=True)

def rerun(ad, W, seed, k):
    st = compute_states(ad, seed)
    comp, oh, S = composition(W, st)
    lab, _, _ = fit_final(features(comp, oh), k, seed)
    reg = ad.obs["Region"].values
    return {"seed": seed, "n_states": len(S), "n_niches": int(len(set(lab))), "ARI": float(ARI(reg, lab)), "NMI": float(NMI(reg, lab))}

In [7]:
seed_all(SEED)
ad = load_smi(DATA_PATH)
if SMOKE_TEST:
    ad = ad[np.sort(np.random.RandomState(SEED).choice(ad.n_obs, min(SMOKE_N, ad.n_obs), replace=False))].copy()
print("cells used:", ad.n_obs)

/tmp/ipykernel_2661/3545897795.py:36: FutureWarning: Import from `scanpy.io` instead
  ad = sc.read_h5ad(path)


raw shape (18895, 980) | obs columns: ['cell_type', 'niche', 'cx', 'cy', 'cx_g', 'cy_g', 'merge_cell_type', 'celltype_refined', 'batch', 'region', 'Region']
after filtering (18895, 960) | batches: 4
cells used: 18895


In [8]:
states = compute_states(ad, SEED)
n_states = len(set(states))
print("number of cell states:", n_states)
if not 15 <= n_states <= 25:
    print("WARNING: states outside 15-25; adjust STATE_RES in CONFIG")
pd.DataFrame({"state": states}, index=ad.obs_names).to_csv(f"{OUT_DIR}/states.csv", index_label="cell")
W = spatial_knn(ad.obs[["cx_g", "cy_g"]].values, ad.obs["batch"].values, K_NEIGHBORS)
comp, oh, S = composition(W, states)
X = features(comp, oh)
print("composition features:", X.shape)

	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
Reach convergence after 2 iteration(s).


/tmp/ipykernel_2661/4273479891.py:30: FutureWarning: The argument use_rep is deprecated and will be removed in the future. Use `use` instead.
  sc.pp.neighbors(a, n_neighbors=k, use_rep="X_h", random_state=seed)


number of cell states: 9
composition features: (18895, 9)


In [9]:
k_sel, cand = select_k(X, SEED)
cand.to_csv(f"{OUT_DIR}/k_selection_candidates.csv", index=False)
plot_selection(cand, k_sel, f"{OUT_DIR}/k_selection_curves.png")
print("selected k:", k_sel)

{'k': 4, 'bic': np.float64(388513.1196740509), 'silhouette': np.float64(0.08295862599219278), 'stability': 0.5227115187813235}
{'k': 5, 'bic': np.float64(328768.15313490963), 'silhouette': np.float64(0.041075943796732406), 'stability': 0.9735932002206594}
{'k': 6, 'bic': np.float64(320008.15533713077), 'silhouette': np.float64(0.04550166747682581), 'stability': 0.6523768094318474}
{'k': 7, 'bic': np.float64(316669.9327722282), 'silhouette': np.float64(0.05116112390988146), 'stability': 0.6288351150310759}
{'k': 8, 'bic': np.float64(301617.21857206937), 'silhouette': np.float64(0.03722085150940085), 'stability': 0.6757580806110808}
{'k': 9, 'bic': np.float64(290627.6428167773), 'silhouette': np.float64(0.059776226774705836), 'stability': 0.7567336672050808}
{'k': 10, 'bic': np.float64(288532.4206790125), 'silhouette': np.float64(0.08289858769111574), 'stability': 0.7466624066471322}
selected k: 5


In [10]:
labels, post, merges = fit_final(X, k_sel, SEED)
print("niches after min-size merge:", len(set(labels)), "| merges:", merges)
pd.DataFrame({"niche": labels, "state": states}, index=ad.obs_names).to_csv(f"{OUT_DIR}/labels.csv", index_label="cell")
ev0 = evaluate(labels, ad.obs["Region"].values)
print("ARI", round(ev0["ARI"], 3), "NMI", round(ev0["NMI"], 3))

niches after min-size merge: 5 | merges: {}
ARI 0.104 NMI 0.22


/tmp/ipykernel_2661/3023037174.py:19: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  summ = pd.DataFrame({"n_cells": ct.sum(1), "frac_cells": ct.sum(1) / ct.values.sum(),
/tmp/ipykernel_2661/3023037174.py:19: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  summ = pd.DataFrame({"n_cells": ct.sum(1), "frac_cells": ct.sum(1) / ct.values.sum(),
/tmp/ipykernel_2661/3023037174.py:20: Pandas4Warning: Starting with pandas version 4.0 all arguments of max will be keyword-only.
  "majority_region": ct.idxmax(1), "purity": ct.max(1) / ct.sum(1)})
/tmp/ipykernel_2661/3023037174.py:20: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  "majority_region": ct.idxmax(1), "purity": ct.max(1) / ct.sum(1)})


In [11]:
rob = []
for s in ROBUST_SEEDS:
    if s == SEED:
        rob.append({"seed": s, "n_states": n_states, "n_niches": int(len(set(labels))), "ARI": ev0["ARI"], "NMI": ev0["NMI"]})
    else:
        rob.append(rerun(ad, W, s, k_sel))
    print(rob[-1])
rdf = pd.DataFrame(rob); rdf.to_csv(f"{OUT_DIR}/seed_robustness.csv", index=False)
robust = {"seeds": ROBUST_SEEDS, "per_seed": rob}
for c in ["ARI", "NMI", "n_niches"]:
    robust[c + "_mean"] = float(rdf[c].mean()); robust[c + "_sd"] = float(rdf[c].std(ddof=1)) if len(rdf) > 1 else 0.0

{'seed': 0, 'n_states': 9, 'n_niches': 5, 'ARI': 0.10371722436374155, 'NMI': 0.220365734419173}
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
Reach convergence after 2 iteration(s).


/tmp/ipykernel_2661/4273479891.py:30: FutureWarning: The argument use_rep is deprecated and will be removed in the future. Use `use` instead.
  sc.pp.neighbors(a, n_neighbors=k, use_rep="X_h", random_state=seed)


{'seed': 1, 'n_states': 10, 'n_niches': 5, 'ARI': 0.13447864464158396, 'NMI': 0.2364013522317731}
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
Reach convergence after 2 iteration(s).


/tmp/ipykernel_2661/4273479891.py:30: FutureWarning: The argument use_rep is deprecated and will be removed in the future. Use `use` instead.
  sc.pp.neighbors(a, n_neighbors=k, use_rep="X_h", random_state=seed)


{'seed': 42, 'n_states': 10, 'n_niches': 5, 'ARI': 0.13342272739576147, 'NMI': 0.24307593604944389}


In [12]:
sil_niche = float(silhouette_score(X, labels, sample_size=min(SIL_SAMPLE, len(X)), random_state=SEED)) if len(set(labels)) > 1 else None
sel_row = cand[cand["k"] == k_sel].iloc[0]
metrics = {
    "n_niches": int(len(set(labels))), "ARI": ev0["ARI"], "NMI": ev0["NMI"], "silhouette": sil_niche,
    "n_states": n_states, "stability": {"selected_k_bootstrap_ARI": float(sel_row["stability"]), "mean_posterior": float(post.mean())},
    "seed_robustness": robust, "baseline": BASELINE,
    "selection_record": {"rule": "mean of min-max scaled BIC-elbow, silhouette and bootstrap ARI over k; argmax; Region not used",
                         "k_selected": k_sel, "candidates": cand.to_dict("records"),
                         "min_size_merges_gmm_component_ids": merges, "state_resolution": STATE_RES},
    "note": "ARI/NMI/purity are reporting only; computed after selection",
}
config = {k: globals()[k] for k in CONFIG_KEYS}
config["versions"] = {}
for p in ["scanpy", "anndata", "pandas", "numpy", "scikit-learn", "leidenalg", "igraph", "harmony-pytorch"]:
    try: config["versions"][p] = version(p)
    except PackageNotFoundError: config["versions"][p] = None
config["python"] = sys.version
config["runtime_sec"] = time.time() - T0
config["notes"] = ["Wilcoxon one-vs-rest on log-normalised raw counts", "states and niches are anonymous integer strings", "composition = neighbour state fractions, per-batch kNN excluding self"]
stab = pd.DataFrame({"mean_posterior": pd.Series(post).groupby(labels).mean()})
ev = save_contract(OUT_DIR, ad, labels, {"state": states}, stab, metrics, config)
print(ev["summary"])

/tmp/ipykernel_2661/3023037174.py:19: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  summ = pd.DataFrame({"n_cells": ct.sum(1), "frac_cells": ct.sum(1) / ct.values.sum(),
/tmp/ipykernel_2661/3023037174.py:19: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  summ = pd.DataFrame({"n_cells": ct.sum(1), "frac_cells": ct.sum(1) / ct.values.sum(),
/tmp/ipykernel_2661/3023037174.py:20: Pandas4Warning: Starting with pandas version 4.0 all arguments of max will be keyword-only.
  "majority_region": ct.idxmax(1), "purity": ct.max(1) / ct.sum(1)})
/tmp/ipykernel_2661/3023037174.py:20: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  "majority_region": ct.idxmax(1), "purity": ct.max(1) / ct.sum(1)})


       n_cells  frac_cells          majority_region    purity
niche                                                        
0         5246    0.277640                   stroma  0.290698
1         4829    0.255570  myeloid-enriched stroma  0.485194
2         4234    0.224080                   stroma  0.434813
3         4090    0.216459  myeloid-enriched stroma  0.730318
4          496    0.026250           tumor interior  1.000000


In [13]:
nc = niche_composition(comp, labels, S)
nc.to_csv(f"{OUT_DIR}/niche_composition.csv", index=False)
sm = top_markers(ad, states, "state")[["state", "rank", "gene", "log2FC"]]
sm.to_csv(f"{OUT_DIR}/state_markers.csv", index=False)
print(nc.head(10)); print(sorted(os.listdir(OUT_DIR)))

  niche state  frac_in_niche  frac_global  fold_enrichment  rank
0     0     0       0.316533     0.117917         2.684383     1
1     0     6       0.145393     0.132658         1.096004     2
2     0     3       0.124508     0.123595         1.007381     3
3     0     4       0.072652     0.081616         0.890171     4
4     0     1       0.088614     0.107005         0.828122     5
5     1     4       0.109167     0.081616         1.337568     1
6     1     1       0.140740     0.107005         1.315261     2
7     1     5       0.111983     0.089459         1.251778     3
8     1     6       0.154552     0.132658         1.165047     4
9     1     2       0.207621     0.188260         1.102842     5
['crosstab_vs_region.csv', 'k_selection_candidates.csv', 'k_selection_curves.png', 'labels.csv', 'metrics.json', 'niche_composition.csv', 'niche_markers_long.csv', 'niche_markers_wide.csv', 'niche_summary.csv', 'niche_vs_truth.png', 'run_config.json', 'seed_robustness.csv', 'state_mar

In [14]:
config["runtime_sec"] = time.time() - T0
jdump(config, f"{OUT_DIR}/run_config.json")
shutil.make_archive("/content/m1_composition_outputs", "zip", OUT_DIR)
try:
    from google.colab import files
    files.download("/content/m1_composition_outputs.zip")
except ImportError:
    print("zip saved at /content/m1_composition_outputs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>